In [ ]:
import pandas as pd
import time
from datetime import datetime, timedelta
from gnews import GNews
import random
import os
from googlenewsdecoder import gnewsdecoder

# manually mapping outlets to their regional group & country
outlets_dict = { 
    "nytimes.com": {"region": "US", "country": "US"}, # left
    "apnews.com": {"region": "US", "country": "US"}, # left
    "cnn.com": {"region": "US", "country": "US"}, # left leaning
    "washingtonpost.com": {"region": "US", "country": "US"},
    "newsweek.com": {"region": "US", "country": "US"}, # center
    "san.com": {"region": "US", "country": "US"}, # center 
    "nypost.com": {"region": "US", "country": "US"}, # right leaning
    "washingtonexaminer.com": {"region": "US", "country": "US"}, # right leaning
    "foxnews.com": {"region": "US", "country": "US"}, # right
    "spectator.org": {"region": "US", "country": "US"}, # right

    "politiken.dk": {"region": "greenland & denmark", "country": "denmark"}, # denmark
    "cphpost.dk": {"region": "greenland & denmark", "country": "denmark"},
    "thelocal.dk": {"region": "greenland & denmark", "country": "denmark"},
    "berlingske.dk": {"region": "greenland & denmark", "country": "denmark"},
    "jyllands-posten.dk": {"region": "greenland & denmark", "country": "denmark"},
     "borsen.dk": {"region": "greenland & denmark", "country": "denmark"},
    "altinget.dk": {"region": "greenland & denmark", "country": "denmark"}, 
    "dr.dk": {"region": "greenland & denmark", "country": "denmark"},
    "seven59.dk": {"region": "greenland & denmark", "country": "denmark"},
    "sermitsiaq.ag": {"region": "greenland & denmark", "country": "greenland"}, # greenland
    "knr.gl": {"region": "greenland & denmark", "country": "greenland"},
        
    "thelocal.fr": {"region": "affected europe", "country": "france"}, # france
    "lemonde.fr": {"region": "affected europe", "country": "france"},
    "france24.com": {"region": "affected europe", "country": "france"},
    "connexionfrance.com": {"region": "affected europe", "country": "france"},
    "thelocal.de": {"region": "affected europe", "country": "germany"}, # germany
    "spiegel.de": {"region": "affected europe", "country": "germany"},
     "welt.de": {"region": "affected europe", "country": "germany"},
    "zeit.de": {"region": "affected europe", "country": "germany"},
    "faz.net": {"region": "affected europe", "country": "germany"},
    "dw.com": {"region": "affected europe", "country": "germany"},
    "nltimes.nl": {"region": "affected europe", "country": "netherlands"}, # netherlands
    "dutchnews.nl": {"region": "affected europe", "country": "netherlands"},
    "thelocal.no": {"region": "affected europe", "country": "norway"}, # norway
    "newsinenglish.no": {"region": "affected europe", "country": "norway"},
    "tnp.no": {"region": "affected europe", "country": "norway"},
     "aftenbladet.no": {"region": "affected europe", "country": "norway"},
    "arctictoday.com": {"region": "affected europe", "country": "norway"},
    "en.highnorthnews.com": {"region": "affected europe", "country": "norway"},
    "norwaynews.com": {"region": "affected europe", "country": "norway"},
    "hs.fi": {"region": "affected europe", "country": "finland"}, # finland
    "helsinkitimes.fi": {"region": "affected europe", "country": "finland"},
    "yle.fi": {"region": "affected europe", "country": "finland"},
    "finlandtoday.fi": {"region": "affected europe", "country": "finland"},
    "thisisfinland.fi": {"region": "affected europe", "country": "finland"},
    "suomikhaber.fi": {"region": "affected europe", "country": "finland"},
    "theguardian.com": {"region": "affected europe", "country": "uk"}, # uk
    "bbc.com": {"region": "affected europe", "country": "uk"},
    "telegraph.co.uk": {"region": "affected europe", "country": "uk"},
    "dailymail.co.uk": {"region": "affected europe", "country": "uk"},
    "skynews.com": {"region": "affected europe", "country": "uk"},
    "aftenbladet.se": {"region": "affected europe", "country": "sweden"}, # sweden 
    "thelocal.se": {"region": "affected europe", "country": "sweden"},
    "swedenherald.com": {"region": "affected europe", "country": "sweden"},

    "cbc.ca": {"region": "other", "country": "canada"}, # canada
    "ctvnews.ca": {"region": "other", "country": "canada"},
    "globalnews.ca": {"region": "other", "country": "canada"},
    "theglobeandmail.com": {"region": "other", "country": "canada"},
    "news.com.au": {"region": "other", "country": "australia"}, # australia
    "abc.net.au": {"region": "other", "country": "australia"}, 
    "9news.com.au": {"region": "other", "country": "australia"},
    "7news.com.au": {"region": "other", "country": "australia"},
     "smh.com.au": {"region": "other", "country": "australia"},
    "ndtv.com": {"region": "other", "country": "india"}, # india
    "india.com": {"region": "other", "country": "india"},
    "timesofindia.indiatimes.com": {"region": "other", "country": "india"},
    "indiatoday.in": {"region": "other", "country": "india"},
}

folder_name = "scraped_outlets"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)

def scrape_master_automated(outlets, start_date, end_date):
    google_news = GNews(language="en", max_results=100)
    # GNews python package searches the Google News RSS feed
    for domain, info in outlets.items():
        safe_name = domain.replace(".", "_")
        file_path = f"{folder_name}/{safe_name}.csv" 
        # create separate csv files for every outlet
        if os.path.exists(file_path):
            print(f"skipping {domain}")
            continue
            
        print(f"\nscraping {domain} at {datetime.now().strftime('%H:%M:%S')}")
        # keeping track of the outlet being scraped
        outlet_data = []
        curr = start_date
        
        while curr < end_date:
            nxt = curr + timedelta(days=14)
            # 2 week rolling period
            google_news.start_date = curr
            google_news.end_date = nxt
            
            try:
                results = google_news.get_news(f"Greenland site:{domain}") 
                if results: # query-based search
                    for a in results:
                        try:
                            decoded = gnewsdecoder(a['url'], interval=1)
                            # GNews saves urls by its source, therefore gnewsdecoder is required: a python package 
                            # that decodes google news urls to their original ones
                            real_url = decoded.get("decoded_url")

                            if decoded.get("status") and domain in real_url:
                                outlet_data.append({
                                    "title": a.get("title"),
                                    "url": real_url,
                                    "date": a.get("published date"),        
                                    "region": info["region"],
                                    "country": info["country"],             # code was frequently stopped and rerun per every few 
                                    "domain": domain                        # outlets as blockages often occurred causing 0 articles 
                                    })                                      # to automatically be returned
                                    
                        except:                                             # besides pauses and long breaks, network connection was often
                            continue                                        # switched per every few outlets to mitigate the chance of IP
                                                                            # blockages: wi-fi, mobile data hotspot and vpn were used
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                print(f"  error in chunk for {domain}: {e}")
                time.sleep(10)
            # pauses to avoid IP blockages or detection of suspicious activity
            curr = nxt
            
        if outlet_data: # save to csv
            df_outlet = pd.DataFrame(outlet_data)
            df_outlet.to_csv(file_path, index=False)
            print(f"finished {domain}: {len(df_outlet)} articles saved")
        else:
            print(f"finished {domain}: no articles found")

        print(f"waiting 20 minutes")
        time.sleep(1200) 
        # break to avoid IP blockages

start_date_thesis = datetime(2025, 1, 1)
end_date_thesis = datetime(2026, 2, 28)

scrape_master_automated(outlets_dict, start_date_thesis, end_date_thesis)

# the following sites were used to yield the necessary code for collecting articles with the GNews package
# https://pypi.org/project/gnews/
# https://pypi.org/project/googlenewsdecoder/

In [ ]:
# GNews' query-based search often collects articles with semantic similarity, therefore an additional verification
# process is implemented to handle the noisy data with keyword-based eliminationn

input_folder = "scraped_outlets"
output_folder = "relevant_regions"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def relevant(title):
    title = str(title).lower()
    keywords = ["greenland", "greenlandic", "greenland's", "greenlander", "greenlanders", "greenlander's", "nuuk"]
    # keywords broad enough to include all articles relevant to the crisis
    has_country = any(word in title for word in keywords)
    return has_country

regions_mapping = {
    "US": [],
    "affected europe": [],
    "greenland & denmark": [],
    "other": []
}
# now mapping each outlet file to their designated regions to output 4 region-based csv files
for domain, info in outlets_dict.items():
    reg = info['region']
    if reg in regions_mapping:
        regions_mapping[reg].append(domain)

for region_name, domains in regions_mapping.items():
    print(f"\nverifying region {region_name.upper()}")
    
    regional_list = []
    
    for domain in domains:
        safe_name = domain.replace(".", "_")
        file_path = f"{input_folder}/{safe_name}.csv"
        if os.path.exists(file_path):
            df_temp = pd.read_csv(file_path)
            regional_list.append(df_temp)
    
    if not regional_list:
        print(f"none found for {region_name}")
        continue

    df = pd.concat(regional_list, ignore_index=True) # combine per region
    initial_count = len(df)

    df['clean_title'] = df['title'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.strip() # cleaning

    df = df.drop_duplicates(subset=['clean_title'])
    df = df.drop_duplicates(subset=['url'])
    after_dedup = len(df)
    df_verified = df[df['title'].apply(relevant)].copy()
    # avoid duplicate articles or urls
    
    final_count = len(df_verified)
    
    safe_region_name = region_name.replace(" & ", "_").replace(" ", "_")
    output_file = f"{output_folder}/{safe_region_name}_verified.csv"
    # new regional csv files
    
    df_verified.drop(columns=['clean_title']).to_csv(output_file, index=False)

    print(f"original articles: {initial_count}")
    print(f"removed: {initial_count - after_dedup}")
    print(f"removed by keyword filtering: {after_dedup - final_count}")
    print(f"final verified count for {region_name}: {final_count}")
    # keeping track of amount of articles removed

In [ ]:
# full text extraction with newspaper3k
# https://newspaper.readthedocs.io/en/latest/

from newspaper import Article
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

input_folder = "relevant_regions"     
output_folder = "final_text_datasets"   

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def extract_text(url):
    try:
        article = Article(url, fetch_images=False, memoize_articles=False) 
        article.download()              # set memoize=False to avoid caching of old/broken versions
        article.parse()
        return article.text
    except Exception:
        return None

verified_files = [f for f in os.listdir(input_folder) if f.endswith("_verified.csv")]

for file_name in verified_files:
    print(f"\nextracting text from {file_name}")
    
    file_path = os.path.join(input_folder, file_name)
    df = pd.read_csv(file_path)
    
    if 'full_text' in df.columns: # avoid redownloading
        print(f"  {file_name} text already exists")
        continue

    urls = df['url'].tolist()
    results = []

    # download 10 articles at a time (increase speed of run)
    with ThreadPoolExecutor(max_workers=10) as executor:
        results = list(tqdm(executor.map(extract_text, urls), total=len(urls), desc="Downloading")) # tqdm creates a progress bar to 
                                                                                                    # stay up to date with the process
    df['full_text'] = results
    
    initial_count = len(df)
    df = df.dropna(subset=['full_text'])                        # remove rows where extraction failed
    df = df[df['full_text'].str.strip().str.len() > 100]        # remove texts with < 100 words
    
    final_count = len(df)
    
    output_name = file_name.replace("_verified.csv", "_FINAL_TEXT.csv")         # create new csv files (keep old for progress tracking)
    output_path = os.path.join(output_folder, output_name)
    df.to_csv(output_path, index=False)
    
    print(f"finished {file_name}")
    print(f"successfully extracted {final_count} / {initial_count} articles")

In [ ]:
# intermediate dataset inspection
df_us = pd.read_csv("final_text_datasets/other_CLEAN.csv")      # manually change csv file name accordingly
df_us_all = pd.read_csv("relevant_regions/other_verified.csv")
print(df_us['domain'].value_counts())

In [ ]:
# next stage: manual noise identification & removal
TARGET_FILE = "final_text_datasets/other_FINAL_TEXT.csv"
OUTPUT_FILE = TARGET_FILE.replace("_FINAL_TEXT.csv", "_CLEAN.csv")

def clean_text(text):
    if not isinstance(text, str) or len(text) < 10:
        return ""
    
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()] # split into paragraphs
    
    cleaned_paragraph = []
    seen_paragraph = set()

    
    noise = [                                  
    'not been edited by ndtv staff',
    'published from a syndicated feed',
    'vishnu som reports from',
    'published by: satyam singh',
    'tune in',
    
    'the audio version of this article is generated by ai',
    'mispronunciations can occur',
    'watch article share options',
    'send this page to someone via email',
    '2:12 47 months ago',
    
    'share this on facebook',
    'twitter send this by email',       # example junk for different media outlet (manually extracted)
    'messenger share this on',
    'read more:',
    'related coverage:',
    
    'advertisement',
    'with inputs from agencies',
    '- ends', 
    'follow us on tiktok',
    'download the cbc news app'
]

    for p in paragraphs:
        p_lower = p.lower()
        
        if p_lower in seen_paragraph:
            continue
        
        if any(n in p_lower for n in noise):
            continue
            
        cleaned_paragraph.append(p)
        seen_paragraph.add(p_lower)

    return "\n\n".join(cleaned_paragraph)

if os.path.exists(TARGET_FILE):
    print(f"loading {TARGET_FILE}...")
    df = pd.read_csv(TARGET_FILE)
    initial_count = len(df)

    df['full_text'] = df['full_text'].apply(clean_text)
    df = df[df['full_text'].str.len() > 100] 
    # remove articles that are too short or completely empty
    df.to_csv(OUTPUT_FILE, index=False)
    
    final_count = len(df)
    print(f"removed {initial_count - final_count} noisy articles")
else:
    print(f"error on '{TARGET_FILE}'")

In [ ]:
# bar chart to depict word count per regional group
import matplotlib.pyplot as plt
import seaborn as sns

files = {
    'US': 'final_text_datasets/US_CLEAN.csv',
    'Affected Europe': 'final_text_datasets/affected_europe_CLEAN.csv',     # gather file paths per region
    'Greenland/DK': 'final_text_datasets/greenland_denmark_CLEAN.csv',
    'Other': 'final_text_datasets/other_CLEAN.csv'
}

stats = []
for region, path in files.items():
    df = pd.read_csv(path)
    total_words = df['full_text'].astype(str).apply(lambda x: len(x.split())).sum() # sum up word counts in full_text column
    stats.append({'region': region, 'total words': total_words})

df_stats = pd.DataFrame(stats).sort_values(by='total words', ascending=False)

# build bar plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.barplot(data=df_stats, x='region', y='total words', palette='viridis')

for i, val in enumerate(df_stats['total words']): # add totals to top of bar chart
    plt.text(i, val, f'{int(val):,}', ha='center', va='bottom', fontweight='bold')

plt.title('total word count per region')
plt.ylabel('word count')
plt.xlabel('region')
plt.savefig('wc_regional.png', dpi=300)

In [ ]:
# keeping track of dataset size per region

raw_folder = "scraped_outlets"
verified_folder = "relevant_regions" 
final_folder = "final_text_datasets"

stats = []

region_files = {
    "US": "US_verified.csv",
    "Affected Europe": "affected_europe_verified.csv",
    "Greenland & Denmark": "greenland_denmark_verified.csv",
    "Other": "other_verified.csv"
}

for region_name, verified_filename in region_files.items():
    
    raw_count = 0 # calculate raw counts
    domains_in_region = [d for d, info in outlets_dict.items() if info['region'] == region_name.lower() or info['region'] == region_name]
    
    for domain in domains_in_region:
        path = os.path.join(raw_folder, domain.replace(".", "_") + ".csv")
        if os.path.exists(path):
            raw_count += len(pd.read_csv(path))
  
    verified_path = os.path.join(verified_folder, verified_filename) # verified counts
    verified_count = len(pd.read_csv(verified_path)) if os.path.exists(verified_path) else 0
    
    final_filename = verified_filename.replace("_verified.csv", "_CLEAN.csv") # full text counts
    final_path = os.path.join(final_folder, final_filename)
    final_count = len(pd.read_csv(final_path)) if os.path.exists(final_path) else 0
    
    stats.append({
        "region": region_name,
        "step 1 search results": raw_count,
        "step 2 verification results": verified_count,
        "step 3 full body text extraction results": final_count,
        "final vs raw (%)": f"{(final_count/raw_count)*100:.1f}%" if raw_count > 0 else "0%"
    })

df_summary = pd.DataFrame(stats)

total_raw = df_summary["step 1 search results"].sum()
total_verified = df_summary["step 2 verification results"].sum()
total_final = df_summary["step 3 full body text extraction results"].sum()

df_summary.loc[len(df_summary)] = ["total", total_raw, total_verified, total_final, f"{(total_final/total_raw)*100:.1f}%"]

print(df_summary.to_string(index=False))

In [ ]:
# stratified undersampling to account for the significant undersampling of greenland & denmark articles

us_leaning_map = { 
    'nytimes.com': 'Left', 'apnews.com': 'Left', 'cnn.com': 'Left',
    'washingtonpost.com': 'Left_Leaning', 'newsweek.com': 'Left_Leaning',
    'san.com': 'Center', 'washingtonexaminer.com': 'Center',
    'nypost.com': 'Right_Leaning',                                  # dictionary that manually maps USA outlets to political leaning to
    'foxnews.com': 'Right', 'spectator.org': 'Right'                # maintain a balanced political spectrum when performing undersampling
}

def balanced_sample(df, col, target):               # df = dataframe, column to perform the undersampling on
    if len(df) <= target:                           # col = column to perform undersampling on
        return df                                   # target = target size to reach in col
    
    groups = df[col].unique()               # gets the unique categories (political leanings) and calculates integer division
    ideal = target // len(groups)           # to find the ideal amount to deduct in order to hit the target
                                           
    sampled_parts = []
    overflow_count = 0 # represents the leftover room fron under-represented groups
    
    for g in groups:
        sub_df = df[df[col] == g]
        if len(sub_df) < ideal:
            sampled_parts.append(sub_df)
            overflow_count += (ideal - len(sub_df))
        else:
            pass
            
    for g in groups:
        sub_df = df[df[col] == g] # isolates rows in each group
        if len(sub_df) >= ideal:                                           # if this group has more rows than the ideal amount, calculates
            n_to_take = ideal + (overflow_count // 2)                      # how much to sample: ideal + portion of overflow / 2
            n_to_take = min(n_to_take, len(sub_df)) # sampled < existing
            sampled_parts.append(sub_df.sample(n=n_to_take, random_state=42))         # randomly selects rows and ensure reproducibility with
                                                                                      # random state = 42
    final_sample = pd.concat(sampled_parts) 
    # combine all into one dataframe

    if len(final_sample) > target:                                  # verification step: if sample is larger than the target, 
        return final_sample.sample(n=target, random_state=42)       # a random sample is taken
    return final_sample

# loading csv files per region
df_gd = pd.read_csv('final_text_datasets/greenland_denmark_CLEAN.csv')          # doesn't get run through balanced_sample

df_us = pd.read_csv('final_text_datasets/US_CLEAN.csv')         
df_us['stratify_col'] = df_us['domain'].map(us_leaning_map)     # stratify_col: new column that maps political affiliation to outlet
df_us_sampled = balanced_sample(df_us, 'stratify_col', 500)

df_ae = pd.read_csv('final_text_datasets/affected_europe_CLEAN.csv')
df_ae_sampled = balanced_sample(df_ae, 'country', 600)                          # target is higher for affected europe to account for
                                                                                # all the countries this group involves
df_other = pd.read_csv('final_text_datasets/other_CLEAN.csv')
df_other_sampled = balanced_sample(df_other, 'country', 500)                    # regions affected & europe balanced by country

# region labels to ensure their groups are known in the new dataset
df_gd['region_label'] = 'greenland+denmark'
df_us_sampled['region_label'] = 'US'
df_ae_sampled['region_label'] = 'affected_europe'
df_other_sampled['region_label'] = 'other'

master_df = pd.concat([df_us_sampled, df_ae_sampled, df_other_sampled, df_gd])
master_df.to_csv('MASTER_SAMPLED.csv', index=False)

print("final counts:")
print(master_df['region_label'].value_counts())

In [ ]:
# regional summaries

df = pd.read_csv('MASTER_SAMPLED.csv')

df['word_count'] = df['full_text_cleaned'].astype(str).apply(lambda x: len(x.split()))

region_stats = df.groupby('region_label').agg(
    article_count=('full_text_cleaned', 'count'),
    total_words=('word_count', 'sum'),
    avg_words_per_article=('word_count', 'mean')
).sort_values(by='article_count', ascending=False)

outlet_stats = df.groupby(['region_label', 'domain']).agg(
    article_count=('full_text_cleaned', 'count'),
    total_words=('word_count', 'sum'),
    avg_words_per_article=('word_count', 'mean')
).sort_values(by=['region_label', 'article_count'], ascending=[True, False])

print("region summary")
print(region_stats)
print("\ntop outlets")
print(outlet_stats.head(10))

region_stats.to_csv('regionstats.csv')
outlet_stats.to_csv('outletstats.csv')